<a href="https://colab.research.google.com/github/asaveraasad-data/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import subprocess

REPO_URL = "https://github.com/asaveraasad-data/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship


## Finding 1: Growing vs. Declining Content

The paper observed that growing pages tended to be younger and longer than declining pages. The authors describe this as an observational comparison rather than evidence that page age or length directly causes growth.

### Methodology question

How was the "growing" versus "declining" label defined, and would the same relationship hold across different clients or different time periods?

### Why this matters

Because the findings are observational, grouped or time-aware validation would help determine whether these patterns generalize beyond the specific portfolio used in the study.


---

## Finding 2: Recently Refreshed Content

The paper observed that recently refreshed older pages generally outperformed older pages that were not refreshed.

### Methodology question

How were pages selected for refreshing, and were refreshed pages comparable to pages that were not refreshed?

### Why this matters

If editors naturally chose their highest-priority pages to refresh, then the observed improvement may partly reflect selection bias rather than the refresh itself. A matched comparison or controlled study would strengthen the evidence.

In [2]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


## My model under an honest split (before/after)

In Week 5, I evaluated my model using a grouped split by `client_id`, which prevents the same client from appearing in both the training and testing sets. To demonstrate why this matters, I also evaluated the same model using a standard random train/test split.

I kept the preprocessing, features, and Logistic Regression model identical so that the only difference is the validation strategy.

Following the FlyRank validation guidance, I also report the base rate of the target class because model performance should always be interpreted relative to a simple baseline.

In [13]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
)

# -------------------------------------------------
# Load data
# -------------------------------------------------

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# Create label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# -------------------------------------------------
# Features
# -------------------------------------------------

feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
]

X = df[feature_cols]
y = df["is_declining_label"]

# -------------------------------------------------
# Base Rate
# -------------------------------------------------

base_rate = y.mean()
majority_baseline = max(base_rate, 1 - base_rate)

print("=" * 50)
print("Base Rate")
print("=" * 50)
print(f"Positive class rate: {base_rate:.3f}")
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

# -------------------------------------------------
# BEFORE: Random Split
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

random_model = LogisticRegression(max_iter=1000)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

random_acc = accuracy_score(y_test, random_pred)

print("\n" + "=" * 50)
print("Random Split")
print("=" * 50)
print(f"Accuracy: {random_acc:.3f}")

# -------------------------------------------------
# AFTER: Honest Grouped Split
# -------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=df["client_id"],
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

group_model = LogisticRegression(max_iter=1000)

group_model.fit(X_train, y_train)

group_pred = group_model.predict(X_test)

group_acc = accuracy_score(y_test, group_pred)

print("\n" + "=" * 50)
print("Grouped Split")
print("=" * 50)
print(f"Accuracy: {group_acc:.3f}")

print("\nClassification Report")
print(classification_report(y_test, group_pred))

Base Rate
Positive class rate: 0.542
Majority-class baseline accuracy: 0.542

Random Split
Accuracy: 0.556

Grouped Split
Accuracy: 0.505

Classification Report
              precision    recall  f1-score   support

           0       0.47      0.11      0.17      3014
           1       0.51      0.89      0.65      3149

    accuracy                           0.51      6163
   macro avg       0.49      0.50      0.41      6163
weighted avg       0.49      0.51      0.42      6163



### Interpretation

The random split achieved an accuracy of **55.6%**, while the grouped split achieved **50.5%**. The grouped evaluation is more realistic because it prevents the model from learning client-specific patterns that could appear in both the training and testing sets.

The majority-class baseline was **54.2%**, meaning the grouped model did not outperform a simple majority prediction. This suggests that the current feature set has limited ability to generalize to unseen clients.

The classification report also shows that the model identifies declining pages more often than non-declining pages, resulting in high recall for the positive class but low recall for the negative class. These results indicate that the current model should be considered **directional evidence** and **decision-support**, rather than a reliable production model.

In [15]:
# -------------------------------------------------
# Failure Analysis
# -------------------------------------------------

errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = group_pred

misclassified = errors[
    errors["Actual"] != errors["Predicted"]
]

columns_to_show = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "Actual",
    "Predicted",
]

misclassified[columns_to_show].head(10)

,days_since_last_update,impressions_90d,ctr,avg_position,Actual,Predicted
13,103,307,0.00,39.8,0,1
36,20,371,1.35,5.4,0,1
44,20,64,0.00,55.8,1,0
56,20,16,0.00,4.6,0,1
60,20,25,4.00,6.2,1,0
64,8,2639,0.11,7.2,0,1
78,92,59,0.00,8.7,0,1
82,13,1810,0.44,8.3,0,1
96,13,1197,0.00,21.4,0,1
126,20,82,0.00,21.0,0,1


### Failure Analysis

I reviewed a sample of the misclassified pages produced under the grouped validation split.

Several errors occurred even when pages had similar values for historical metrics such as impressions, CTR, average position, and days since the last update. For example, some pages with low CTR and many days since the last update were correctly identified as declining, while others with similar characteristics were not. This suggests that the current feature set does not fully capture the factors associated with future content decline.

These failure examples indicate that additional information, such as richer historical trends, content characteristics, or client-specific context, may improve prediction performance. Rather than treating the model as definitive, these results should be interpreted as **measured**, **directional** evidence intended for **decision-support**.

## Leakage Audit

Before trusting the model, I audited the final feature set for possible data leakage using the FlyRank leakage taxonomy.

The audit focused on four questions:

1. Are any features derived from the target label?
2. Do any feature windows overlap the prediction window?
3. Are any product or decision-derived features included?
4. Are identifiers used only for grouping rather than prediction?

The table below summarizes the findings.

In [16]:
import pandas as pd

audit = pd.DataFrame(
    [
        {
            "Feature / Column": "trend_direction",
            "Leakage Risk": "High",
            "Reason": "Used to create the target label (is_declining_label). Never used as a feature."
        },
        {
            "Feature / Column": "trend_pct",
            "Leakage Risk": "High",
            "Reason": "Parent variable used to derive trend_direction. Excluded from training."
        },
        {
            "Feature / Column": "client_id",
            "Leakage Risk": "None (Grouping Only)",
            "Reason": "Used only for GroupShuffleSplit, never as a model feature."
        },
        {
            "Feature / Column": "content_id",
            "Leakage Risk": "None (Identifier)",
            "Reason": "Unique identifier only. Excluded from training."
        },
        {
            "Feature / Column": "days_since_last_update",
            "Leakage Risk": "Low",
            "Reason": "Available before prediction."
        },
        {
            "Feature / Column": "impressions_90d",
            "Leakage Risk": "Low",
            "Reason": "Historical metric used as an input feature."
        },
        {
            "Feature / Column": "ctr",
            "Leakage Risk": "Low",
            "Reason": "Historical click-through rate available before prediction."
        },
        {
            "Feature / Column": "avg_position",
            "Leakage Risk": "Low",
            "Reason": "Historical search performance metric."
        }
    ]
)

display(audit)

,Feature / Column,Leakage Risk,Reason
0,trend_direction,High,Used to create the target label (is_declining_...
1,trend_pct,High,Parent variable used to derive trend_direction...
2,client_id,None (Grouping Only),"Used only for GroupShuffleSplit, never as a mo..."
3,content_id,None (Identifier),Unique identifier only. Excluded from training.
4,days_since_last_update,Low,Available before prediction.
5,impressions_90d,Low,Historical metric used as an input feature.
6,ctr,Low,Historical click-through rate available before...
7,avg_position,Low,Historical search performance metric.


## Claim Rewrite

### Original claim

The Logistic Regression model accurately predicts whether content is declining.

### Revised claim

In this project, the Logistic Regression model **observed** patterns associated with declining content using a small set of historical features. Under grouped validation, the model achieved lower performance than the majority-class baseline, indicating limited generalization to unseen clients. These **measured** results should therefore be interpreted as **directional** evidence for **decision-support**, rather than as proof that the model can reliably predict content decline across all clients.

## Self-check

Before you submit, confirm each line honestly:

✅Every section above is filled — markdown thinking AND the code that backs it

✅The notebook runs top to bottom with no errors (Runtime → Run all)

✅No client names, URLs, or private queries anywhere

✅My claims use careful words: observed, measured, directional, decision-support

✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.